In [1]:
from pathlib import Path
from PIL import Image, ImageOps

# =========================
# HEIC対応（pillow-heif）
# =========================
HEIF_AVAILABLE = False
try:
    import pillow_heif  # pip install pillow-heif
    pillow_heif.register_heif_opener()
    HEIF_AVAILABLE = True
except Exception:
    HEIF_AVAILABLE = False

# =========================
# 設定
# =========================
base_dir = Path(r"C:\Users\yamaz\Documents\GitHub\yamazakilab-ynu.github.io\docs\album")
output_dir = base_dir / "album_resized"
output_dir.mkdir(exist_ok=True)

extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp", ".heic"]
TARGET_WIDTH = 600

print("=== Image Resize + Convert to PNG Start ===")
print(f"📁 base_dir   : {base_dir}")
print(f"📁 output_dir : {output_dir}")
print(f"🧩 HEIC support: {'ON' if HEIF_AVAILABLE else 'OFF (install pillow-heif)'}\n")

# =========================
# メイン処理
# =========================
for img_path in base_dir.rglob("*"):

    # フォルダは除外
    if img_path.is_dir():
        continue

    # ✅ 出力フォルダ配下は絶対に処理しない（無限処理防止）
    try:
        img_path.relative_to(output_dir)
        continue
    except ValueError:
        pass

    # 拡張子チェック
    if img_path.suffix.lower() not in extensions:
        continue

    # 出力は常にPNG（元の相対パスを維持して .png 化）
    rel = img_path.relative_to(base_dir)
    out_path = (output_dir / rel).with_suffix(".png")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # ✅ 既に出力があれば何もしない（強制スキップ）
    if out_path.exists():
        print(f"⏭ Skipped (already resized): {rel}")
        continue

    # HEICなのにサポートが無ければスキップ（インストール案内）
    if img_path.suffix.lower() == ".heic" and not HEIF_AVAILABLE:
        print(f"❌ HEIC not supported (install pillow-heif): {rel}")
        continue

    try:
        with Image.open(img_path) as img:
            # iPhone画像の向き補正
            img = ImageOps.exif_transpose(img)

            # リサイズ（600px以下は拡大しない）
            if img.width <= TARGET_WIDTH:
                resized = img.copy()
            else:
                w_percent = TARGET_WIDTH / float(img.width)
                new_h = int(img.height * w_percent)
                resized = img.resize((TARGET_WIDTH, new_h), Image.LANCZOS)

            # PNG保存用にモード調整（透過も保持）
            if resized.mode == "P":
                resized = resized.convert("RGBA")
            elif resized.mode not in ("RGB", "RGBA"):
                resized = resized.convert("RGBA")

            # PNGで保存
            resized.save(out_path, format="PNG", optimize=True)
            print(f"✅ Saved: {out_path.name}")

    except Exception as e:
        print(f"❌ Error processing {rel}: {e}")

print("\n=== Process Finished ===")


=== Image Resize + Convert to PNG Start ===
📁 base_dir   : C:\Users\yamaz\Documents\GitHub\yamazakilab-ynu.github.io\docs\album
📁 output_dir : C:\Users\yamaz\Documents\GitHub\yamazakilab-ynu.github.io\docs\album\album_resized
🧩 HEIC support: ON

⏭ Skipped (already resized): boston.jpg
⏭ Skipped (already resized): combi.jpg
⏭ Skipped (already resized): contest.jpg
✅ Saved: ISAFM2026-1.png
✅ Saved: ISAFM2026-2.png
⏭ Skipped (already resized): jafoe.jpg
⏭ Skipped (already resized): klimt.jpg
⏭ Skipped (already resized): kotsugi-lab.jpg
⏭ Skipped (already resized): mandm.heic
⏭ Skipped (already resized): mbn-jig.heic
⏭ Skipped (already resized): mbn.heic
⏭ Skipped (already resized): meeting.jpg
⏭ Skipped (already resized): msj2025-1.heic
⏭ Skipped (already resized): msj2025-2.heic
⏭ Skipped (already resized): nanoterasu-1.jpg
⏭ Skipped (already resized): nanoterasu-2.jpg
⏭ Skipped (already resized): nanoterasu-3.jpg
⏭ Skipped (already resized): nanoterasu-4.jpg
⏭ Skipped (already resized):